# Modelagem da Camada Gold: Fato Devoluções

In [ ]:
import sys
from pathlib import Path

# Adiciona o diretório raiz do projeto ao sys.path para importações locais
sys.path.append(str(Path.cwd().parent.parent))

from pyspark.sql import functions as F
from src.modules.spark_session import get_spark_session, close_spark_session
import src.modules.modeling_fato_utils as modeling_fato
import src.modules.utils as utils

In [ ]:
# Inicializa a SparkSession conectada ao cluster do container
spark = get_spark_session("ModelagemGoldFatoDevolucoes")

# Leitura das tabelas da camada Silver

In [ ]:
# Define caminhos das origens na Silver
silver_devolucoes_path = "s3a://silver/devolucoes"

# Lê os dados da Silver definindo como None caso a origem não exista
try:
    df_devolucoes = spark.read.parquet(silver_devolucoes_path)
except Exception as e:
    print(f"Aviso: Tabela Silver de Devoluções não encontrada: {e}")
    df_devolucoes = None

# Colunas de Devoluções na Origem

In [ ]:
# Mostra a pré-visualização das colunas caso a tabela não seja None
if df_devolucoes is not None:
    print("=== Colunas em Silver Devoluções ===")
    display(df_devolucoes.limit(5).toPandas())

# Cria a Fato Devoluções (fato_devolucoes)

In [ ]:
# Executa a lógica de modelagem da fato
df_fato_devolucoes = modeling_fato.create_fato_devolucoes(df_devolucoes)

if df_fato_devolucoes is not None:
    # Exibe informações sobre o DataFrame gerado
    print(f"Quantidade total de registros na fato devoluções: {df_fato_devolucoes.count()}")
    df_fato_devolucoes.printSchema()
    display(df_fato_devolucoes.limit(10).toPandas())
else:
    print("Nenhuma fato de devoluções foi processada (tabela Silver de devoluções estava ausente).")

In [ ]:
# Finaliza a sessão do Spark
close_spark_session(spark)